# Classification de noms avec un RNN caractère par caractère

Vincent Guigue (vincent.guigue@agroparistech.fr), d'après le tutoriel officiel PyTorch de Sean Robertson : [NLP From Scratch: Classifying Names with a Character-Level RNN](https://pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html)

**Tâche** : on dispose de quelques milliers de noms de famille issus de 18 langues ; à partir de l'orthographe d'un nom, prédire sa langue d'origine. Un RNN lit le nom caractère par caractère ; le dernier état caché sert à la classification (*many-to-one*).

```
> Hinton          > Schmidhuber
(-0.47) Scottish  (-0.19) German
(-1.52) English   (-2.48) Czech
(-3.57) Irish     (-2.68) Dutch
```

Avant de démarrer, prenez 2 minutes de recul pour considérer les applications possibles d'un tel travail.

La tâche est résolue dans le tutoriel d'origine, mais avec un code « artisanal » (un exemple à la fois, boucle à la main sur les caractères). L'idée de ce notebook est de la **remettre dans le cadre standard** défini dans les chapitres précédents. Vous êtes encouragés à comparer avec le tutoriel au fur et à mesure.

**Aborder les modifications dans l'ordre proposé :**
1. Données chargées sous forme de listes d'entiers [code fourni], séparation apprentissage / test, `DataLoader`
2. Modèle : utiliser les modules récurrents de PyTorch (`nn.RNN`, `nn.LSTM`...) et une couche d'`Embedding` (même pour du *one-hot*)
3. Batchs de plusieurs noms de longueurs différentes : *padding*
4. Pondération des classes dans les tirages (classes très déséquilibrées)

## Table des matières

- [0. Mise en place : librairies, device, TensorBoard](#0-mise-en-place-librairies-device-tensorboard)
- [A. Préparation des données](#a-preparation-des-donnees)
    - [A.1 Des noms aux tenseurs [code fourni]](#a-1-des-noms-aux-tenseurs-code-fourni)
    - [A.2 Batchs et *padding*](#a-2-batchs-et-padding)
    - [A.3 Pondération des classes](#a-3-ponderation-des-classes)
- [B. Rappel : le module `nn.RNN`](#b-rappel-le-module-nn-rnn)
- [C. Création du réseau](#c-creation-du-reseau)
- [D. Apprentissage](#d-apprentissage)
- [E. Évaluation des résultats](#e-evaluation-des-resultats)
    - [E.1 Qualitative](#e-1-qualitative)
    - [E.2 Quantitative : matrice de confusion](#e-2-quantitative-matrice-de-confusion)
- [F. Exercices](#f-exercices)

<a id="0-mise-en-place-librairies-device-tensorboard"></a>
## 0. Mise en place : librairies, device, TensorBoard

In [ ]:
import glob, os, string, time, unicodedata
from itertools import chain

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

In [ ]:
# ---- Choix du device ----
# Pour ces petits RNN, la récurrence est une boucle séquentielle sur le temps : sur Mac, le GPU (MPS)
# est en général PLUS LENT que le CPU (mesuré sur le TP métro : 12 s/epoch en MPS contre 1,2 s/epoch en CPU).
# MPS n'est donc activé que sur demande. Avec un GPU NVIDIA (CUDA), le gain reste réel.
USE_MPS = False

if torch.cuda.is_available():
    device = torch.device("cuda")
elif USE_MPS and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("torch :", torch.__version__, "| device :", device)
_ = torch.manual_seed(0)  # reproductibilité

In [ ]:
# ---- TensorBoard ----
from pathlib import Path
TB_PATH = Path("/tmp/logs/module4")    # une expérience = un sous-répertoire : SummaryWriter(TB_PATH / nom)
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:                    # tensorboard absent : on continue sans enregistrer les courbes
    class SummaryWriter:
        def __init__(self, *args, **kwargs): pass
        def add_scalar(self, *args, **kwargs): pass
        def close(self): pass
    print("tensorboard non installé : les courbes ne seront pas enregistrées")

# Option 1 (recommandée) : dans un terminal, lancer la commande ci-dessous puis ouvrir http://localhost:6006
print(f"tensorboard --logdir {TB_PATH}")

# Option 2 : TensorBoard directement dans le notebook (décommenter les 2 lignes)
# %load_ext tensorboard
# %tensorboard --logdir {TB_PATH}

<a id="a-preparation-des-donnees"></a>
## A. Préparation des données

Le répertoire `data/names` contient 18 fichiers `[Langue].txt`, un nom par ligne, la plupart romanisés (il faut tout de même convertir l'Unicode en ASCII).

On construit un dictionnaire `{langue: [noms...]}`. Les variables génériques *category* et *line* (langue et nom ici) sont conservées pour l'extensibilité.

In [ ]:
# Pas de modification importante dans le chargement des données brutes.
# Noter l'ajout de deux caractères spéciaux qui seront repris ensuite : PAD (padding) et SOS (start of sequence).
# Les noms sont évocateurs... Si vous ne comprenez pas à quoi ils servent : demandez !

all_letters = string.ascii_letters + " .,';_%"
PAD = len(all_letters) - 1   # '%'
SOS = len(all_letters) - 2   # '_' (optionnel, non utilisé ici)
n_letters = len(all_letters)

def unicodeToAscii(s):
    """ Unicode => ASCII, merci à https://stackoverflow.com/a/518232/2809427 """
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn' and c in all_letters)

print(unicodeToAscii('Ślusàrski'))

def readLines(filename):
    lines = open(filename, encoding='utf-8').read().strip().split('\n')
    return [unicodeToAscii(line) for line in lines]

category_lines, all_categories = {}, []   # dictionnaire {langue : liste de noms} et liste des langues
for filename in sorted(glob.glob('data/names/*.txt')):
    category = os.path.splitext(os.path.basename(filename))[0]
    all_categories.append(category)
    category_lines[category] = readLines(filename)
n_categories = len(all_categories)

print(n_categories, "langues :", all_categories)
print(category_lines['Italian'][:5])

<a id="a-1-des-noms-aux-tenseurs-code-fourni"></a>
### A.1 Des noms aux tenseurs [code fourni]

Le tutoriel d'origine code chaque lettre en *one-hot* (`<1 x n_letters>`). On se contente ici de coder chaque nom par la **liste des indices** de ses lettres : la transformation en vecteurs sera faite dans le réseau par une couche d'`Embedding` (partie C).

In [ ]:
##################
# MODIFICATION 1 : données => listes d'entiers [code fourni]
##################
def letterToIndex(letter):
    return all_letters.find(letter)

def lineToTensor(line):
    return torch.tensor([letterToIndex(c) for c in line], dtype=torch.long)

def tensorToName(x):   # vérification de la réversibilité
    return "".join(all_letters[i] for i in x)

print('J :', letterToIndex('J'))
print('Jones :', lineToTensor('Jones'), "=>", tensorToName(lineToTensor('Jones')))

In [ ]:
##################
# MODIFICATION 1 : Dataset + séparation apprentissage / test [code fourni]
##################
alllab = torch.tensor([i for i, c in enumerate(all_categories) for _ in category_lines[c]], dtype=torch.long)
allx   = [lineToTensor(name) for c in all_categories for name in category_lines[c]]

class NameDataset(Dataset):
    def __init__(self, input_tensors, label_tensors):
        assert len(input_tensors) == len(label_tensors), "Les listes d'entrée et d'étiquettes doivent avoir la même longueur"
        self.input_tensors = input_tensors
        self.label_tensors = label_tensors

    def __len__(self):
        return len(self.input_tensors)

    def __getitem__(self, idx):
        return self.input_tensors[idx], self.label_tensors[idx]

name_ds = NameDataset(allx, alllab)
# 90 % apprentissage, 10 % test ; graine fixée => même découpage à chaque exécution
train_dataset, test_dataset = random_split(name_ds, [0.9, 0.1], generator=torch.Generator().manual_seed(0))
print("tailles :", len(train_dataset), len(test_dataset))

<a id="a-2-batchs-et-padding"></a>
### A.2 Batchs et *padding*

Pour former un batch, les noms (de longueurs différentes) doivent être complétés avec le caractère `PAD` jusqu'à la longueur du plus long nom du batch : c'est le rôle de la fonction `collate_fn` passée au `DataLoader`. On renvoie aussi les **longueurs réelles**, indispensables pour retrouver le dernier « vrai » état caché de chaque séquence.

Convention du module : séquences *temps en premier* => le batch de noms est de taille `longueur_max x batch`.

In [ ]:
##################
# MODIFICATION 3 : Batch + padding
##################
def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded_sequences = pad_sequence(sequences, batch_first=False, padding_value=PAD)   # Lmax x batch
    return padded_sequences, lengths, torch.tensor(labels)

<a id="a-3-ponderation-des-classes"></a>
### A.3 Pondération des classes

Les classes sont très déséquilibrées (près de la moitié des noms sont russes !). Un classifieur qui répond toujours « Russian » a déjà une bonne accuracy... On peut **rééquilibrer les tirages** : un `WeightedRandomSampler` tire les exemples (avec remise) avec une probabilité proportionnelle à un poids.

Construire le poids de chaque exemple d'apprentissage pour que toutes les classes soient tirées aussi souvent (en moyenne).

In [ ]:
##################
# MODIFICATION 4 : Pondérer les classes
##################
train_labels = torch.stack([train_dataset[i][1] for i in range(len(train_dataset))])

# CONSTRUIRE UN ALGORITHME DE PONDERATION DES ECHANTILLONS (inversement proportionnel à l'effectif de la classe)
##  TODO 

train_sampler = WeightedRandomSampler(weights=train_sample_weights, num_samples=len(train_labels), replacement=True)

In [ ]:
Bsize = 128
# MODIF 1 : un nom à la fois (utile pour débugger : pas de padding)
train_loader_1 = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
# MODIF 3 : batchs + padding
train_loader = DataLoader(train_dataset, batch_size=Bsize, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=Bsize, shuffle=False, collate_fn=collate_fn)
# MODIF 4 : tirages pondérés (sampler et shuffle sont incompatibles)
train_loader_w = DataLoader(train_dataset, batch_size=Bsize, sampler=train_sampler, collate_fn=collate_fn)

# Test : observer un batch de taille 1 puis un batch de taille 4 (où est le padding ?)
for loader in [train_loader_1, DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)]:
    x, lengths, y = next(iter(loader))
    print("x :", x.size(), "\n", x, "\nlongueurs :", lengths, "| labels :", y)

<a id="b-rappel-le-module-nn-rnn"></a>
## B. Rappel : le module `nn.RNN`

Exemple du cours : `seq_h` contient tous les états cachés (`T x B x h`), `last_h` le dernier état de chaque couche (`num_layers x B x h`).

In [ ]:
model = nn.RNN(input_size=100, hidden_size=50, num_layers=1, nonlinearity='tanh', bias=True)
seq_x = torch.rand(23, 10, 100)   # T x B x d
seq_h, last_h = model(seq_x)      # pour de la classification, on utilise last_h
print("seq_h : ", seq_h.size())
print("last_h: ", last_h.size())

<a id="c-creation-du-reseau"></a>
## C. Création du réseau

1. **Embedding one-hot** : une couche `nn.Embedding(n_letters, n_letters)` dont les poids sont la matrice identité (et une ligne nulle pour `PAD`) transforme un indice en vecteur one-hot. Ces poids ne doivent pas être appris : désactiver le gradient. (`nn.Embedding.from_pretrained` fait tout cela en une ligne.)
2. **Module récurrent** (`nn.RNN`, puis tester `nn.LSTM` / `nn.GRU`) et **décodeur** linéaire.
3. **Forward** : avec du padding, le « dernier » état caché `last_h` correspond à la dernière position **paddée**, pas au dernier vrai caractère ! Deux solutions :
    - aller chercher à la main `seq_h[lengths[i]-1, i]` pour chaque séquence `i` ;
    - (recommandé) emballer la séquence avec [`pack_padded_sequence`](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html) : le module récurrent s'arrête alors au bon endroit pour chaque séquence et `last_h` est correct (et le calcul est plus rapide).

In [ ]:
class RNN(nn.Module):
    def __init__(self, hidden_size, output_size, n_letters, cell="rnn"):
        super().__init__()
        self.hidden_size, self.cell = hidden_size, cell

        # 1. Embeddings one-hot, figés (pas de gradient), ligne PAD à 0
        ##  TODO 

        # 2. self.rec = module récurrent (RNN, LSTM ou GRU selon cell) ; self.h2o = décodeur
        ##  TODO 

    def forward(self, input, lengths=None):
        """ input : Lmax x batch (indices) ; lengths : longueurs réelles => logits : batch x n_categories """
        # 1. indices => vecteurs one-hot
        # 2. passage dans le module récurrent (en gérant le padding)
        # 3. prédiction à partir du dernier état caché
        ##  TODO 
        return output, hidden

In [ ]:
print("n_letters:", n_letters, "| n_categories:", n_categories)
n_hidden = 128
rnn = RNN(n_hidden, n_categories, n_letters)

# test sur un nom (batch de taille 1)
input = lineToTensor('Albert').unsqueeze(1)   # L x 1
output, hidden = rnn(input)
print("input:", input.size(), "=> output:", output.size())

# test sur un batch + loss
loss = nn.CrossEntropyLoss()
x, lengths, y = next(iter(train_loader))
yhat, hidden = rnn(x, lengths)
print("x:", x.size(), "=> yhat:", yhat.size(), "| loss:", loss(yhat, y).item())

La sortie est un tenseur `batch x n_categories` de **logits** : plus la valeur est élevée, plus la catégorie est probable (le softmax est inclus dans `CrossEntropyLoss`).

<a id="d-apprentissage"></a>
## D. Apprentissage

Boucle standard : la seule différence avec les TP précédents est que le `DataLoader` renvoie un triplet `(x, lengths, y)`. On suit l'accuracy **et** l'accuracy équilibrée (moyenne des rappels par classe), plus pertinente quand les classes sont déséquilibrées.

In [ ]:
from sklearn.metrics import balanced_accuracy_score

def evaluate(model, loader):
    """ renvoie loss, accuracy, accuracy équilibrée, (y, prédictions) """
    model.eval()
    cumloss, Y, P = 0., [], []
    with torch.no_grad():
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            yhat, _ = model(x, lengths)
            cumloss += loss(yhat, y).item() * len(y)
            Y.append(y.cpu()); P.append(yhat.argmax(1).cpu())
    Y, P = torch.cat(Y), torch.cat(P)
    return cumloss / len(Y), (Y == P).float().mean().item(), balanced_accuracy_score(Y, P), (Y, P)


def train(model, epochs, train_loader, test_loader, name):
    writer = SummaryWriter(TB_PATH / f"{name}-{time.strftime('%Y%m%d-%H%M%S')}")
    model = model.to(device)
    optim = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in tqdm(range(epochs), desc=name):
        model.train()
        cumloss, count = 0., 0
        for x, lengths, y in train_loader:
            x, y = x.to(device), y.to(device)
            optim.zero_grad()
            yhat, _ = model(x, lengths)
            l = loss(yhat, y)
            l.backward()
            optim.step()
            cumloss += l.item() * len(y)     # .item() : on ne garde pas le graphe de calcul !
            count += len(y)
        writer.add_scalar("loss/train", cumloss / count, epoch)
        l, acc, bacc, _ = evaluate(model, test_loader)
        writer.add_scalar("loss/test", l, epoch)
        writer.add_scalar("accuracy/test", acc, epoch)
        writer.add_scalar("balanced_accuracy/test", bacc, epoch)
    return model

Apprendre deux modèles identiques : l'un avec des tirages uniformes (`train_loader`), l'autre avec des tirages pondérés (`train_loader_w`). Comparer les courbes dans TensorBoard.

In [ ]:
# ~1 min par modèle sur CPU
n_epoch = 30
rnn   = train(RNN(n_hidden, n_categories, n_letters), n_epoch, train_loader,   test_loader, "names-rnn")
rnn_w = train(RNN(n_hidden, n_categories, n_letters), n_epoch, train_loader_w, test_loader, "names-rnn-weighted")

<a id="e-evaluation-des-resultats"></a>
## E. Évaluation des résultats

<a id="e-1-qualitative"></a>
### E.1 Qualitative

Écrire une fonction `predict(model, name, k)` qui affiche les `k` langues les plus probables pour un nom.

In [ ]:
def predict(model, name, k=3):
    ##  TODO 

for name in ["Satoshi", "Dovesky", "Jackson", "Guigue", "Schmidhuber"]:
    predict(rnn, name)

<a id="e-2-quantitative-matrice-de-confusion"></a>
### E.2 Quantitative : matrice de confusion

Calculer, **sur le test**, l'accuracy, l'accuracy équilibrée et la matrice de confusion (normalisée par ligne) des deux modèles ; `sklearn.metrics` fournit tout le nécessaire.

Quel est l'effet de la pondération des tirages ? Quel modèle choisir ?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (model, title) in zip(axes, [(rnn, "tirages uniformes"), (rnn_w, "tirages pondérés")]):
    ##  TODO 
plt.tight_layout()
plt.show()
##  TODO 

<a id="f-exercices"></a>
## F. Exercices

- Comparer `nn.RNN`, `nn.LSTM` et `nn.GRU` (argument `cell`) ; augmenter la taille de l'état caché, empiler plusieurs couches (`num_layers`).
- Remplacer les embeddings one-hot figés par des embeddings appris (`nn.Embedding(n_letters, d)` avec d < n_letters) : qu'est-ce que ça change ?
- Autre façon de traiter le déséquilibre : pondérer la loss (`nn.CrossEntropyLoss(weight=...)`) au lieu des tirages.
- Tester d'autres jeux de données *séquence => catégorie* : mot => langue, prénom => genre, titre => rubrique...